In [ ]:
%pip install -e ../../../
%pip install requests
%pip install geopandas
%pip install shapely
%pip install geopy
%pip install plotly
%pip install nbformat

In [ ]:
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12807)
import pandas as pd

from water_futures_battle.core.base_model import StaticProperties

In [ ]:
# let's create for each type x random sources so that the total production matches what was written in Wevin 2022 report
# sources_sps = StaticProperties('sources-static_properties')

def source_id_gen(t: str, x: int) -> str:
    return f"S{t[0].upper()}{x:04d}"

SOURCE_TYPES = ['groundwater', 'surface_water', 'desalination']

In [ ]:
# let's start groundwater sources:
import geopandas as gpd
import zipfile

def extract_kmz_coordinates(kmz_path):
    # Ensure your GeoPandas setup supports the KML driver (usually does)
    with zipfile.ZipFile(kmz_path, 'r') as kmz:
        # KMZ files typically contain a 'doc.kml' file
        kml_data = kmz.read('doc.kml')
    gdf = gpd.read_file(kml_data, driver='KML')
    
    all_points = []
    
    # Iterate through each feature (Placemark) in the KML file
    for index, row in gdf.iterrows():
        geometry = row['geometry']
        
        # Check if the geometry is a LineString (a path)
        if geometry.geom_type == 'LineString':
            
            # The .coords property gives you an iterable of (longitude, latitude, altitude) tuples
            # KML standard is Longitude, Latitude, Altitude (X, Y, Z)
            points = list(geometry.coords)
            
            # You can store the points along with the Placemark's name or other attributes
            all_points.extend([{
                'longitude': lon,
                'latitude': lat
            } for lon, lat, _ in points])

    coordinates_dict = {
        'latitude': [pt['latitude'] for pt in all_points],
        'longitude': [pt['longitude'] for pt in all_points]
    }
    return coordinates_dict

sources_dfs = []
for source_type in SOURCE_TYPES:
    sources_coordinates = extract_kmz_coordinates(f"../raw-data/{source_type}.kmz")
    df = pd.DataFrame(sources_coordinates)


    # Pre-allocate the elvation column, we will assign it 3 meters above the municipality
    df['elevation'] = np.nan

    df['source_type'] = source_type
    df['counter'] = np.array(range(1, len(df)+1))

    sources_dfs.append(df)


# Unfortunately, we missed one source in Fryslan
sources_dfs.append(
    pd.DataFrame([{
        "latitude": 53.48317,
        "longitude": 6.16511,
        "elevation": np.nan,
        "source_type": 'groundwater',
        "counter": 212
    }])
)

sources_df = pd.concat(sources_dfs, ignore_index=True)

sources_df['source_id'] = sources_df[['source_type', 'counter']].apply(
    lambda row: source_id_gen(row['source_type'], int(row['counter'])),
    axis=1
)

# We also know the names of a few surface water production locations:
SW_PL_NAMES = {
    "SS0001": "Heel",
    "SS0002": "Gat van de Kerksloot",
    "SS0003": "Brakel",
    "SS0004": "Haringvliet",
    "SS0005": "Bergambacht",
    "SS0006": "Inlaat Nieuwegein",
    "SS0007": "Bethunpolder",
    "SS0008": "Amsterdam-Rijnkanaal",
    "SS0009": "Inlaat Andijk & Waterwinstation Prinses Juliana",
    "SS0010": "De Punt"
}
sources_df['name'] = sources_df['source_id'].map(lambda x: SW_PL_NAMES.get(x, ""))

# A few recurrent selection masks
MASK_GW = sources_df['source_type'] == 'groundwater'
MASK_SW = sources_df['source_type'] == 'surface_water'
MASK_DS = sources_df['source_type'] == 'desalination'

ISLAND_SOURCES_IDS = ['SG0208', 'SG0209', 'SG0210','SG0211', 'SG0212']
MASK_ISLANDS = sources_df['source_id'].isin(ISLAND_SOURCES_IDS)

In [ ]:
from water_futures_battle.core import Settings
from water_futures_battle.jurisdictions import build_state
settings_desc = {
    "start_year": 2000,
    "end_year": 2024,
    "national_budget": 0,
    "lifeline_volume": 80,
    "seed": 128
}
nl_desc = {
    'name': 'Netherlands',
    'id': 'NL0000',
    'jurisdictions-static_properties': 'jurisdictions-static_properties.xlsx',
    'municipalities-dynamic_properties': 'municipalities-dynamic_properties.xlsx'
}
netherlands = build_state(
    nl_desc,
    data_path='../../jurisdictions/',
    settings=Settings.from_config(settings_desc)
    )

# Upload the demands generated in the `jurisdicitons-view_demands-historical` notebook
pvs_max_daily_dem_df = pd.read_csv('../../jurisdictions/provinces-max_daily_demands-CMD.csv', index_col='year')
juris_max_hourly_dem_df = pd.read_csv('../../jurisdictions/jurisdictions-max_hourly_demands-CMH.csv', index_col='year', parse_dates=True)
juris_max_hourly_dem_df.index = juris_max_hourly_dem_df.index.year

In [ ]:
# Assume that given the latitude and longitude we can infer in which province it is.
# Otherwise we can use some map, e.g., https://simplemaps.com/gis/country/nl#admin1
from shapely.geometry import Point, shape
import warnings
# Use the official Dutch national projection system: RD New
PROJECTED_CRS = 'EPSG:28992'

def get_province_name(row, a_gdf: gpd.GeoDataFrame) -> str:
    """
    Given a longitude and latitude, finds the province it belongs to 
    in the GeoDataFrame.
    """
    latitude: float = row['latitude']
    longitude: float = row['longitude']

    # Create a shapely Point object from the coordinates
    # GeoJSON coordinates are (longitude, latitude)
    target_point = Point(longitude, latitude)
    
    # Check which feature (province) contains the point
    # The 'contains' method returns a boolean Series
    # We use .any(axis=1) for MultiPolygons which have multiple parts
    # The `geodataframe.geometry` column contains the Polygon/MultiPolygon shapes
    
    # Find the row(s) where the geometry contains the target point
    # We use a mask to filter the GeoDataFrame
    mask = a_gdf.contains(target_point)
    
    # The mask should result in one or zero matches
    if mask.any():
        # Get the 'name' property of the matching feature(s)
        # .iloc[0] is used to get the first (and hopefully only) match
        return a_gdf.loc[mask, 'name'].iloc[0]
    else:
        # 2. No strict containment found: Find the closest province
        # Reproject the coordinates 
        original_crs = a_gdf.crs
        gdf_proj = a_gdf.to_crs(PROJECTED_CRS)
        target_point_gs = gpd.GeoSeries([target_point], crs=original_crs).to_crs(PROJECTED_CRS).iloc[0]
        
        # Calculate the distance from the target point to ALL province geometries
        # The result is a pandas Series of distances
        distances = gdf_proj.geometry.distance(target_point_gs)
        
        # Find the index of the minimum distance
        closest_index = distances.idxmin()
        
        # Retrieve the name of the closest province using the index
        closest_province_name = str(gdf_proj.loc[closest_index, 'name'])
        
        # Issue a warning instead of a print/error
        warning_message = (
            f"Point ({latitude}, {longitude}) not found in any province. "
            f"Returning closest province: '{closest_province_name}'. "
            "This may indicate the point is slightly outside the boundaries."
        )
        warnings.warn(warning_message, UserWarning)
        
        return closest_province_name

def name_to_cbs_id(a_name: str, a_state) -> str:
    if len(a_name) == 0:
        return "PVxx"
    if a_name == "Friesland":
        return "PV21"
    for province in a_state.provinces:
        if province.outfiles_name.startswith(a_name.lower().replace('-', '_')[0:9]): # at least noord- because we have noord_holland and noord_brabant
            return province.cbs_id
    
    raise RuntimeError(f"Impossible to match the province : {a_name}")

gdf = gpd.read_file('../raw-data/nl.json')

sources_df['provinces_names'] = sources_df.apply(get_province_name, args=(gdf,), axis=1)
sources_df['province'] = sources_df['provinces_names'].apply(name_to_cbs_id, args=(netherlands,))
sources_df.drop(columns=['provinces_names'], inplace=True)

# Decide which one to keep

We want to keep one at random between all the sources connecting at each node.
The chosen one (*keep*) will keep its activation and closure dates.
For the ones with a closure date and that have not been selected, we repurpose them to be used in 2025
(*repurpose*).
The remaining ones (*reset*), that have not been selected and do not connect to a municipality
that is closed in 2025, we set their dates to None again so that can be still used in the future.

In [ ]:
# now we see which is the closest municipality in 2025 to each source (of type groundwater and surface_water)
# Multiple sources will connect to the same municipality and in those cases 
# we will keep only one at random so that we can use the others in the future as 
# new one that can open
from geopy.distance import geodesic

def closest_muni_finder(a_source_desc, y2025: bool = False) -> str:
    province = netherlands.province(a_source_desc['province'])

    municipalities_distances = {}
    if ((a_source_desc['source_type'] == 'desalination') or y2025):
        municipalities = province.active_municipalities(when=2025)
    else:
        municipalities = province.municipalities
        municipalities = sorted(municipalities, key=lambda m: m.cbs_id)
        # Remove renamed municipalities so that we don't risk selecting on or 
        # the other.
        remove_ms = [m.effective_entity(when=m.end_date) for m in municipalities if m.end_reason == 'renamed' ]
        municipalities = [m for m in municipalities if m not in remove_ms]

    for m in municipalities:
        distance = geodesic(
                    (a_source_desc['latitude'], a_source_desc['longitude']),
                    m.coordinates
                )
        municipalities_distances[m.cbs_id] = distance.km

    closest_municipality = sorted(
        municipalities_distances,
        key=lambda cbs_id: municipalities_distances[cbs_id],
    )[0]
    
    return closest_municipality

LIFT2NEW = "LIFT2NEW"
def closure_date_logic_groundwater(m: str) -> str:
    """
    This function should only be called on groundwater sources.
    Takes the cbs id of a municipality and applies the logic to decide if that source
    should close or not.
    The logic is as follow:
    - municipality doesn't close -> source doesn't close either
    - municipality close:
        - municipality gets renamed -> source doesn't close (it will connect automatically to the new one)
        - municipality gest lifted:
            - to an already existing municipality -> source closes, its capacity will be absorbed by the other sources
            - to a new municipality -> the decision happens at global level, 
    """
    municipality = netherlands.municipality(m)

    # Municipality doesn't close
    if pd.isna(municipality.end_date):
        return ""
    
    # Municipality closes, but because it gets renamed
    if municipality.end_reason == 'renamed':
        return ""
    
    # Municipality gets lifted, but to an already existing one:
    destination_municipality = municipality.effective_entity(when=municipality.end_date)
    if destination_municipality.begin_date < municipality.end_date:
        return municipality.end_date.strftime('%Y-%m-%d')
    
    # Municipality gets lifted to a new one (destination_municipality.begin_date == municipality.end_date)
    # Pass placeholder because the logic needs to happen at global level
    return LIFT2NEW

sources_df['closest_municipality'] = sources_df.apply(lambda row: closest_muni_finder(row), axis=1)
sources_df['activation_date'] = sources_df['closest_municipality'].apply(lambda m: netherlands.municipality(m).begin_date.strftime('%Y-%m-%d'))
sources_df['closure_date'] = sources_df.apply(
    lambda row: "" if row['source_type'] != 'groundwater' else closure_date_logic_groundwater(row['closest_municipality']),
    axis=1
)

def select_groundwater(df: pd.DataFrame) -> pd.DataFrame:
    # Group by closest_municipality
    grouped = df[df['source_type'] == 'groundwater'].groupby('closest_municipality')
    keep_idx = []
    repur_idx = []
    reset_idx = []

    for cm_id in sorted(grouped.groups.keys()):
        group = grouped.get_group(cm_id)
        
        # Pick one at random to keep
        chosen = group.sample(1, random_state=42).index[0]
        keep_idx.append(chosen)
        # For the rest
        for idx in group.index:
            if idx == chosen:
                continue
            # If closure_date is present (not empty): repurpose
            if df.loc[idx, 'closure_date'] not in [None, '', 'NaT']:
                repur_idx.append(idx)
            else:
                reset_idx.append(idx)

    # Reset activation/closure dates for DESALINATION
    mask = df['source_type'] == 'desalination'
    df.loc[mask, ['activation_date', 'closure_date']] = None

    # Reset activation/closure dates for unused sources
    df.loc[reset_idx, ['activation_date', 'closure_date']] = None

    # Repurposed sources are reconnected in 2025
    df['repurposed'] = None
    df.loc[repur_idx, 'repurposed'] = 1
    df.loc[repur_idx, ['activation_date', 'closure_date']] = None
    df.loc[repur_idx, 'closest_municipality'] = df.loc[repur_idx, :].apply(lambda row: closest_muni_finder(row, y2025=True), axis=1)
    # df.drop(columns=['repurposed'], inplace=True)

    return df

# Remains to how to handle the LIFT2NEW sources
def handle_lift2new(df: pd.DataFrame) -> pd.DataFrame:
    # We work on the lift2new sources, and group them by effective cbs_id of their municipality
    # if the new source has one already, they can all close and add in the capacity merge column
    # the name of these closing municipalities.
    # If the new municipality doesn't have any, and only one sources would be there, that source
    # doesn't close and the new municipality inherits this source.
    # If the new municipality doesn't have any, and more than one source is available,
    # we select one at random that will be opened when the new municipality opens and will have
    # capacity equal to the sum of all the capacities of the closing one (plus some factor)
    sources_df['replaces'] = ""
    lifted2new = sources_df['closure_date'] == LIFT2NEW
    sources_df['effective_id'] = sources_df['closest_municipality'].apply(
        lambda m: (
            "" if pd.isna(netherlands.municipality(m).end_date)
            else netherlands.municipality(m).effective_entity(netherlands.municipality(m).end_date).cbs_id
        )
    )

    grouped = sources_df.loc[lifted2new, :].groupby('effective_id')

    for effective_id in sorted(grouped.groups.keys()):
        df = grouped.get_group(effective_id)

        # at least one source is already decided that it will connect there
        mask_eid = (sources_df['closest_municipality'] == effective_id) & (MASK_GW)
        if sum(mask_eid) > 0:
            # thus close all of them
            sources_df.loc[df.index, 'closure_date'] = sources_df.loc[df.index, 'closest_municipality'].apply(
                lambda x: netherlands.municipality(x).end_date.strftime('%Y-%m-%d')
            )
            # and save for that source which capacity will it inherit
            mergecapof = ';'.join(map(str, df.index))
            
            # Should not be more than one with activation_date, but it could be that non has it
            mask_eid_notna = sources_df.loc[mask_eid, 'activation_date'].notna()
            eid_active_df = sources_df.loc[mask_eid, :].loc[mask_eid_notna, :]
            assert len(eid_active_df) <= 1
            if len(eid_active_df) != 0:
                chosen_idx = eid_active_df.index[0]
            else:
                chosen_idx = sources_df.loc[mask_eid,:].sample(1, random_state=42).index[0]
                sources_df.loc[chosen_idx, 'activation_date'] = netherlands.municipality(effective_id).begin_date.strftime('%Y-%m-%d')
                
            sources_df.loc[chosen_idx, 'replaces'] = mergecapof

        # Only one source going to the new municipality
        elif len(df) == 1:
            # thus keep it open
            sources_df.loc[df.index, 'closure_date'] = ""
            
        else:
            # More sources being inherti from the new municipality
            # thus, keep n-1 and the chosen will have the merge capacity of applied
            chosen_idx = df.sample(1, random_state=42).index[0]
            other_idxs = [i for i in df.index if i != chosen_idx]
            sources_df.loc[other_idxs, 'closure_date'] = sources_df.loc[other_idxs, 'closest_municipality'].apply(
                    lambda x: netherlands.municipality(x).end_date.strftime('%Y-%m-%d')
                )
            sources_df.loc[chosen_idx, 'replaces'] = ';'.join(map(str, other_idxs))
            sources_df.loc[chosen_idx, 'activation_date'] = netherlands.municipality(effective_id).begin_date.strftime('%Y-%m-%d')
            sources_df.loc[chosen_idx, 'closure_date'] = ""

    return sources_df

sources_df = select_groundwater(sources_df)
sources_df = handle_lift2new(sources_df)

# Manage the repurposed and reset (usually in big municipalities in 2025)
def check_repurposed(m: str) -> str:
    if sum(sources_df['closest_municipality'] == m) == 1:
        municipality = netherlands.municipality(m)
        return municipality.begin_date.strftime('%Y-%m-%d')
    else:
        return ""
    
MASK_2SIZE = MASK_GW & pd.isna(sources_df['activation_date'])
sources_df.loc[MASK_2SIZE, 'activation_date'] = sources_df.loc[MASK_2SIZE, 'closest_municipality'].apply(
    check_repurposed
)

sources_df['activation_date'] = pd.to_datetime(sources_df['activation_date'])
sources_df['closure_date'] = pd.to_datetime(sources_df['closure_date'])

# Manual fixes for locations and dates

- the map consider the area below the Lek (Amedie Meerkerk and Leerdam) to be part of Zuid holland, but it is actually Utrecht.
- groningen sources don't open or close
- two very big surface water sources in Utrecht province are actually intake locations for water that gets treated for Amsterdam and neighbour municipalites. The one in Nieuwegein specifically see N.V. Watertransportmaatschappij Rijn-Kennemerland (WRK). Waternet and PWN maintain the pipes together and check that everything is working properly. PWN manages the water from the IJsselmeer. Waternet manages the water from the Lekkanaal. (Waternet website).
The other one in the Ijsselmeeer we merge it as one big production location given that there is another close by.
- Similarly, Zeeeland takes its water from # "Gat van de Kerksloot",

In [ ]:
# Intake Nieuwegein to AMS
sources_df.loc[sources_df['source_id'] == 'SS0006', 'latitude'] = 52.388398
sources_df.loc[sources_df['source_id'] == 'SS0006', 'longitude'] = 4.748117
sources_df.loc[sources_df['source_id'] == 'SS0006', 'elevation'] = 2
sources_df.loc[sources_df['source_id'] == 'SS0006', 'name'] = "Inlaat Nieuwegein - treated"
sources_df.loc[sources_df['source_id'] == 'SS0006', 'closest_municipality'] = "GM0363"
sources_df.loc[sources_df['source_id'] == 'SS0006', 'province'] = "PV27"
sources_df.loc[sources_df['source_id'] == 'SS0006', 'activation_date'] = pd.to_datetime("1957-01-01")

# other Utrecht SW to NoordHolland
sources_df.loc[sources_df['source_id'] == 'SS0007', 'closest_municipality'] = "GM0384"
sources_df.loc[sources_df['source_id'] == 'SS0007', 'province'] = "PV27"

# other Utrecht SW to NoordHolland
sources_df.loc[sources_df['source_id'] == 'SS0008', 'closest_municipality'] = "GM0402"
sources_df.loc[sources_df['source_id'] == 'SS0008', 'province'] = "PV27"

# Noord Holland (Inlaat Andijk & Waterwinstation Prinses Juliana) to non closing municipality
sources_df.loc[sources_df['source_id'] == 'SS0009', 'closest_municipality'] = "GM0420"

# "Gat van de Kerksloot" from Brabant to Zeeland
sources_df.loc[sources_df['source_id'] == 'SS0002', 'latitude'] = 51.7127744
sources_df.loc[sources_df['source_id'] == 'SS0002', 'longitude'] = 4.6246410
sources_df.loc[sources_df['source_id'] == 'SS0002', 'elevation'] = 0
sources_df.loc[sources_df['source_id'] == 'SS0002', 'name'] = "Gat van de Kerksloot - treated"
sources_df.loc[sources_df['source_id'] == 'SS0002', 'closest_municipality'] = "GM0716"
sources_df.loc[sources_df['source_id'] == 'SS0002', 'province'] = "PV29"
sources_df.loc[sources_df['source_id'] == 'SS0002', 'activation_date'] = pd.to_datetime("1942-01-01")

# "Brakel" from Brabant to Zuid Holland
sources_df.loc[sources_df['source_id'] == 'SS0003', 'closest_municipality'] = "GM0512"
sources_df.loc[sources_df['source_id'] == 'SS0003', 'province'] = "PV28"
sources_df.loc[sources_df['source_id'] == 'SS0003', 'activation_date'] = pd.to_datetime("1985-01-01")

# Groningen
sources_df.loc[sources_df['source_id'] == 'SG0197', 'closest_municipality'] = "GM0014"
sources_df.loc[sources_df['source_id'] == 'SG0197', 'activation_date' ] = pd.to_datetime("1830-01-01")
sources_df.loc[sources_df['source_id'] == 'SG0197', "closure_date"] = pd.NaT

sources_df.loc[sources_df['source_id'] == 'SG0190', 'activation_date' ] = pd.to_datetime("1969-01-01")
sources_df.loc[sources_df['source_id'] == 'SG0190', 'closest_municipality'] = "GM0037"
sources_df.loc[sources_df['source_id'] == 'SG0190', "closure_date"] = pd.NaT

sources_df.loc[sources_df['source_id'] == 'SS0010', 'closest_municipality'] = "GM0014"
sources_df.loc[sources_df['source_id'] == 'SS0010', 'activation_date' ] = pd.to_datetime("1830-01-01")
sources_df.loc[sources_df['source_id'] == 'SS0010', "closure_date"] = pd.NaT

# Error in the map (from Zuid Holland to Utrecht)
sources_df.loc[sources_df['source_id'] == 'SG0083', 'closest_municipality'] = "GM0707"
sources_df.loc[sources_df['source_id'] == 'SG0083', 'province'] = "PV26"

sources_df.loc[sources_df['source_id'] == 'SG0092', 'closest_municipality'] = "GM0620"
sources_df.loc[sources_df['source_id'] == 'SG0092', 'province'] = "PV26"

# Overijssel 
# instead of small closing municipality to high non closing one (Oldenzaal GM0173)
sources_df.loc[sources_df['source_id'] == 'SG0147', 'closest_municipality'] = "GM0173"
sources_df.loc[sources_df['source_id'] == 'SG0147', 'activation_date'] = pd.to_datetime("1830-01-01")
sources_df.loc[sources_df['source_id'] == 'SG0147', "closure_date"] = pd.NaT

# Limburg
# there is a patch of high elevations places right of the Meuse

## Elevations

Source elevation defines the reservoir head. 
Therefore we put it at the closest municipality elevation plus 3 meters.
In this way, every pump, even the smallest at the highest flow, should be able to provide the 
30 m of pressure at least at the connecting municipality.

In [ ]:
HEAD_SURPLUS_RESERVOIR = 3.0
sources_df['elevation'] = sources_df['closest_municipality'].apply(
    lambda x: netherlands.municipality(x).elevation+HEAD_SURPLUS_RESERVOIR
)

## Capacities

We need to define the capacities of the existing and closed sources (groundwater and surface water) (see KWR data to infer the distribution of the sizes)

For the groundwater sources used we also need to determine what was their permit (see KWR data to infer the rule)
For the groundwater sources not used yet, we only need to define their permit (and the user will decide their capacity)

The user will define the capacity for the desalination.

In [ ]:
# Allocate/prepare the column for the data
sources_df['capacity-nominal'] = np.nan

## Surface water

From table 3.1 and the various maps we devise this size of the surface water production location.
These numbers gives you more or less the 494 M m^3/year produced in the netherlands.

We size them manually given the small number of production locations.

In [ ]:
# Surface water production locations' nominal capacity definition
SW_PL_SIZES = {
    "SS0001": 25, # "Heel",
    "SS0002": 29, # "Gat van de Kerksloot",
    "SS0003": 42, # "Brakel",
    "SS0004": 40, # "Haringvliet",
    "SS0005": 34, # "Bergambacht",
    "SS0006": 38, # "Inlaat Nieuwegein",
    "SS0007": 40, # "Bethunpolder",
    "SS0008": 41, # "Amsterdam-Rijnkanaal",
    "SS0009": 39, # "Inlaat Andijk & Waterwinstation Prinses Juliana",
    "SS0010": 12 # "De Punt",
}

for key, a_size in SW_PL_SIZES.items():
    MASK_2SIZE = (sources_df['source_id'] == key)
    sources_df.loc[MASK_2SIZE, 'capacity-nominal'] = a_size*1_000_000.0/365.0

## Groundwater

For groundwater is different. We will open and close them similarly to municipalities.

In [ ]:
# Groundwater production locations' nominal capacity definition

# Closed groundwater sources had a capacity enough to satisfy the small municipality to which they were connected to
MASK_2SIZE = (MASK_GW &
        pd.notna(sources_df['activation_date']) &
        (pd.notna(sources_df['closure_date']) | MASK_ISLANDS)
)
sources_df.loc[MASK_2SIZE, 'capacity-nominal'] = sources_df.loc[MASK_2SIZE, 'closest_municipality'].apply(
    lambda x: round(juris_max_hourly_dem_df[x].max())*24 # m^3/day
)

# Opening groundwater sources that replace others, are x % bigger than their sum
def merge_demands(x: str) -> float:
    REPLACE_FACTOR = 1.3 # 30%
    replaced_sources = [int(i) for i in str(x).split(';') if str(i).strip() != '']
    return sources_df.loc[replaced_sources, 'capacity-nominal'].sum()*REPLACE_FACTOR

MASK_2SIZE = (
    MASK_GW &
    ~MASK_ISLANDS &
    pd.notna(sources_df['activation_date']) &
    sources_df['replaces'].apply(lambda x: isinstance(x, str) and x.strip() != '')
)
sources_df.loc[MASK_2SIZE, 'capacity-nominal'] = sources_df.loc[MASK_2SIZE, 'replaces'].apply(
    lambda x: merge_demands(x) # Should already be in m^3/day
)

In [ ]:
# Manually set some demands before runnign the algorithm

# Groundwater source in Zeeland
sources_df.loc[sources_df['source_id'] == 'SG0050',
    'capacity-nominal'] = 11e6/365 #  Mm^3/year to m^3/day

# Gelderland 
# Groesbeek has weridly high elevation and two muncipalites joining. Thus we need enough water to cover them
# The muncipality gets laso renamed
groesbeek_area = ['GM0241', 'GM0282', 'GM0265', 'GM1945']
sources_df.loc[sources_df['source_id'] == 'SG0076',
    'capacity-nominal'] = juris_max_hourly_dem_df[groesbeek_area].sum(axis=1).max()*24 # to m^3/day
    
# Winterswijk and a few other are way higher, so we need to make sure that it has enoguh water 
# to sustain themselves because water will not come up from below without help
gelderland__fcap = ['SG0113', 'SG0107','SG0115', 'SG0116', 'SG0103', 'SG0104']
MASK_2SIZE = sources_df['source_id'].isin(gelderland__fcap)
sources_df.loc[MASK_2SIZE,
    'capacity-nominal'] = sources_df.loc[MASK_2SIZE, 'closest_municipality'].apply(
        lambda x: round(juris_max_hourly_dem_df[x].max())*24 # m^3/day
)

# Overijssel
# Oldenzaal is way higher so its source should feed the area
oldenzaal_area = ['GM0173', 'GM0168', 'GM0153', 'GM0158']
sources_df.loc[sources_df['source_id'] == 'SG0147',
    'capacity-nominal'] = juris_max_hourly_dem_df[oldenzaal_area].sum(axis=1).max()*24 # to m^3/day

# Limburg
simpelveld_area = ['GM0965', 'GM0981', 'GM1729', 'GM0936', 'GM0905', 'GM1903', 'GM0994']
sources_df.loc[sources_df['source_id'] == 'SG0007',
    'capacity-nominal'] = juris_max_hourly_dem_df[simpelveld_area].sum(axis=1).max()*24 # to m^3/day

voerendaal_area = ['GM0986', 'GM0917', 'GM0928', 'GM0882', 'GM0899']
sources_df.loc[sources_df['source_id'] == 'SG0006',
    'capacity-nominal'] = juris_max_hourly_dem_df[voerendaal_area].sum(axis=1).max()*24 # to m^3/day

In [ ]:
# Now that we have generated the nominal capacity for the sources that close and
# for those that open at their place, we can compute the capacities of the remaining
# to match the total demand.
  
# We know the total demand to meet thanks to the notebook calculating the historical
# demands.
# Our target capacity for 2030 is 5% more than the highest recorded.
PV_TARGET_CAPACITIES_Y2030__CMD = {
    pv: 1.05*pvs_max_daily_dem_df[pv.cbs_id].max() for pv in netherlands.provinces
}

PV_TARGET_GW_CAPACITY_Y2030__CMD = {pv: (
    max( # no negative target capcities
        tc-sources_df.loc[
                (sources_df['province'] == pv.cbs_id), 'capacity-nominal'
            ].sum(skipna=True),
        0
    )
) for pv, tc in PV_TARGET_CAPACITIES_Y2030__CMD.items()}

# Every province has a different groundwater surplus that can still be tapped
# We do this to represent more thirsty provinces and to incentives the water transfer
PV_GW_AVAILABILITY_FACTOR_MAP = {
    'PV20': 1.1, # Groningen
    'PV21': 1.3, # Fryslan
    'PV22': 1.05, # Drenthe
    # Oost-Nederland
    'PV23': 1.01, # Overijssel
    'PV24': 1.2, # Flevoland
    'PV25': 1.2, # Gelderland
    # West-Nederland
    'PV26': 1.4, # Utrecht
    'PV27': 1.1, # Noord-Holland
    'PV28': 1.1, # Zuid-Holland
    'PV29': 2, # Zeeland
    # Zuid-Nederland
    'PV30': 1.5, # Noord-Brabant
    'PV31': 2, # Limburg
}

PV_REAL_GW_CAPACITY_Y2030__CMD = {
    pv: gtc*PV_GW_AVAILABILITY_FACTOR_MAP[pv.cbs_id]
    for pv, gtc in PV_TARGET_GW_CAPACITY_Y2030__CMD.items()
}

MAX_GW_SIZE = 25e6/365.0 # 25 M m^3/year in m^3/day

PV_RNGS = RNG.spawn(n_children=len(netherlands.provinces))
for i, pv in enumerate(sorted(netherlands.provinces, key=lambda pv: pv.cbs_id)):
    capacity_CMD = PV_REAL_GW_CAPACITY_Y2030__CMD[pv]
    # Get the gw sources without capacity and with activation date
    MASK_PV_GW = (
        MASK_GW & 
        (sources_df['province'] == pv.cbs_id)
    )
    MASK_PV_GW_2SIZE = (
        MASK_PV_GW & 
        pd.notna(sources_df['activation_date']) &
        pd.isna(sources_df['capacity-nominal'])
    )
    n_sources_2size = sum(MASK_PV_GW_2SIZE)

    if n_sources_2size == 0:
        continue

    # Create for each source in the province a random number so that we will have 
    # reproducible numbers (if we use only mask the sizes could change base on 
    # the previous steps, invalidating all following work)
    PV_RNG = PV_RNGS[i]
    SHARES_OF_DEM = PV_RNG.uniform(0.66, 0.9, size=len(MASK_PV_GW))

    proposed_capacities = SHARES_OF_DEM[MASK_PV_GW_2SIZE] * sources_df['closest_municipality'][MASK_PV_GW_2SIZE].apply(
        lambda x: min(
            round(juris_max_hourly_dem_df[x].max())*24, # m^3/day
            MAX_GW_SIZE
        )
    ).to_numpy()
    too_bigs = proposed_capacities > MAX_GW_SIZE
    assert all(~ too_bigs)

    # Now in a do-while fashion, see how much capacity we are missing and spread
    # it equally across the gw sources. If a source would exceed the max, skip it 
    # and in the following iteration we adjust with the new missing capacity.
    missing_pv_capacity = max(capacity_CMD - sum(proposed_capacities), 0)
    while missing_pv_capacity > 0:
        
        # There is still some capacity to assign, we try to spread it equally 
        # across the sources
        additional_source_cap = missing_pv_capacity / n_sources_2size

        # Safety check: if there is no way to assign the missing capacity uniformly
        # to the sources because they would all go above the limit, warn me and 
        # end the loop earlier
        exceeding_if_added = (proposed_capacities+additional_source_cap) > MAX_GW_SIZE
        if all( exceeding_if_added ):
            warnings.warn(f"Province {pv.display_name} ({pv.cbs_id}) can't provide this capacity with these settings")
            break

        proposed_capacities[~exceeding_if_added] = (
            proposed_capacities[~exceeding_if_added] + additional_source_cap
        )

        # Update the missing capacity for the next loop
        missing_pv_capacity = max(capacity_CMD - sum(proposed_capacities), 0)


    too_bigs = proposed_capacities > MAX_GW_SIZE
    assert all(~ too_bigs)
    
    sources_df.loc[MASK_PV_GW_2SIZE, 'capacity-nominal'] = proposed_capacities

## Visualization of the capacities



In [ ]:
import plot_utils

plot_utils.plot_net_capacity_by_elevation(
    netherlands,
    sources_df,
    juris_max_hourly_dem_df
).show()

plot_utils.plot_capacity_distribution(
    sources_df
).show()

plot_utils.plot_capacity_across_time(
    sources_df,
    pvs_max_daily_dem_df.sum(axis=1)
).show()

plot_utils.plot_capacity_across_time_per_prov(
    netherlands,
    sources_df,
    pvs_max_daily_dem_df,
    juris_max_hourly_dem_df
).show()

# Opex volumetric energy factor

Energy factor for the volumetric component of the opex cost.

Depends on the site technology, but every technology has it's own type...

In [ ]:
SEC = {
    "groundwater": (0.1,0.4),
    "surface_water": (0.40, 0.50),
    "desalination": (4, 5),
}

sources_df['opex-volum-energy_factor'] = np.nan

OVEF_RNG = RNG.spawn(n_children=len(SOURCE_TYPES))
for i, source_type in enumerate(SOURCE_TYPES):
    MASK_2SIZE = (sources_df['source_type'] == source_type) & sources_df['activation_date'].notna()

    sources_df.loc[MASK_2SIZE, 'opex-volum-energy_factor'] = np.round(
        OVEF_RNG[i].uniform(
            low=SEC[source_type][0],
            high=SEC[source_type][1],
            size=sum(MASK_2SIZE)
        ), 2
    )

## Permit

Groundwater sources have also a permit.
The permit is expressed in m^3/year

We decided that 30% of the open sources have a permit which is less (between 76 and 100% of the capacity).

For the new ones, we take between 30% and 100% bigger than the permit of the groundwater source that is at the same gw location and is not closed yet capped at 25 M m^3/year.


In [ ]:
sources_df['permit'] = np.nan

PERMIT_BREAKER_SHARE = 0.3 # This percentage of groundwater sources have overextraction possible
MAX_GW_CAPACITY_OVER_PERMIT = 1.3 # At most, a source has been oversized by this much
gw_df = sources_df[MASK_GW & pd.notna(sources_df['capacity-nominal'])]

PERMIT_RNG = RNG.spawn(n_children=1)[0]

permit_breakers = PERMIT_RNG.choice(
            gw_df.index,
            size=int(0.3*len(gw_df)), 
            replace=False
        )
non_permit_breakers = np.setdiff1d(gw_df.index.values, permit_breakers)

permit_factors = PERMIT_RNG.uniform(1/MAX_GW_CAPACITY_OVER_PERMIT, 1, len(permit_breakers))

sources_df.loc[permit_breakers, 'permit'] = sources_df.loc[permit_breakers, 'capacity-nominal']*permit_factors
sources_df.loc[non_permit_breakers, 'permit'] = sources_df.loc[non_permit_breakers, 'capacity-nominal']

sources_df['permit'] = sources_df['permit']*365 # to m^3/year

open_permit_sources_df = sources_df[sources_df['permit'].notna() & sources_df['closure_date'].isna()] # we are already filtering grounwater sources with a permit that do not close by the data release
permits_lookup = open_permit_sources_df.groupby('closest_municipality')['permit'].sum().to_dict() # lookup to see what is the permit associated to each municipality
for pv in sorted(netherlands.provinces, key=lambda x: x.cbs_id):
    new_gw_df = sources_df[MASK_GW & pd.isna(sources_df['permit']) & (sources_df['province']==pv.cbs_id)]
    
    if new_gw_df.empty:
        continue

    target_municipalities = sources_df.loc[new_gw_df.index, 'closest_municipality']

    base_permit_series = target_municipalities.map(permits_lookup)

    random_multipliers = PERMIT_RNG.uniform(1.3, 2.12, size=len(new_gw_df))

    new_permit_values = base_permit_series * random_multipliers

    new_permit_values.clip(upper=25_000_000) # M m^3/year
    
    sources_df.loc[new_gw_df.index, 'permit'] = new_permit_values

    assert sum(MASK_GW & pd.isna(sources_df['permit']) & (sources_df['province']==pv.cbs_id)) == 0, "There are still groundwater sources without a permit!"

## Hydological basin

Surface water plants may have their capacity temporarily reduced because of low river flows (if on a river).

We manage this at the basin level. The basins are: 'Rhine' or 'Meuse'.
No basin means it is independent of the river and doesn't change with it

'Heel' and  'Brakel' are in Meuse (Maas river)
'Inlaat Nieuwegein', 'bethunepolder' and 'Amsterdam-Rijnkanaal' are in 'Rhine'

In [ ]:
sources_df['basin'] = ""
sources_df.loc[sources_df['source_id'].isin(["SS0001", "SS0003"]), 'basin'] = "Meuse"
sources_df.loc[sources_df['source_id'].isin(["SS0006", "SS0007", "SS0008"]), 'basin'] = "Rhine"

## Capacity target
Depends on type:
- Desalination has 0.9
- Surface water has 0.95
- Groundwater has 0.98

In [ ]:
global_df = pd.DataFrame({
    'source_type': SOURCE_TYPES,
    'capacity-target_factor': [0.98, 0.95, 0.9]
})

## Volum tariff increase

Let's say production above the target factor has a volumetric cost that is increase by
Desalination 20%
Surface 15%
Groundwater 10% 
to represent the extra unexpected costs of additional treatment

Example, nominal capacity 1000, target factor 90%
- Case A: production 800 -> cost = "opex-volum-other"*800
- Case B: production 950 -> cost = "opex-volum-other"*800 + "opex-volum-other-multiplier"*"opex-volum-other"*150

In [ ]:
data_map = dict(zip(SOURCE_TYPES, [0.1, 0.15, 0.2]))
global_df['opex-volum-other-multiplier'] = global_df['source_type'].map(data_map) + 1.0

## Construction time

Each of the sources can be constructed, but they take different times:
- groundwater takes 2 to 4 years
- surface water tales 3 to 6 years
- desalination takes 5 to 10 years

In [ ]:
data_map = dict(zip(SOURCE_TYPES, [2, 3, 5]))
global_df['construction_time-min'] = global_df['source_type'].map(data_map)

data_map = dict(zip(SOURCE_TYPES, [4, 6, 10]))
global_df['construction_time-max'] = global_df['source_type'].map(data_map)

In [ ]:
data_map = {t: k[0] for t, k in SEC.items()}
global_df['opex-volum-energy_factor-min'] = global_df['source_type'].map(data_map)

data_map = {t: k[1] for t, k in SEC.items()}
global_df['opex-volum-energy_factor-max'] = global_df['source_type'].map(data_map)

## Max Construction Size Limits

New sources must adhere to maximum size limits determined by their type to ensure realistic planning and development.

- **Surface Water:** New surface water facility capacity cannot exceed a specified amount.

    Controlling variable: `surface_water-max_size-m3/day`)
    Rule: $size<limit$
    Unit: $m^3/\text{day}$
    Decision: We cap it at 60M m^3/year (like the biggest we have in ourt private data). 164.383,56
 
- **Groundwater:** New groundwater facility capacity cannot exceed its issued permit by more than the defined percentage factor.

    Controlling variable: `groundwater-max_permit_surplus`)
    Rule: $size<permit×(1+factor)$
    Unit: dimensionless

- **Desalination:**  New desalination plant capacity cannot exceed a specified amount.

    Controlling variable: `desalination-max_size-m3/day`)
    Rule: $size<limit$
    Unit: $m^3/\text{day}$
    Notes: It will probably be be set to `inf` to reflect unrestricted potential. 

Due to the type-dependent logic and differing units associated with the maximum construction limits, these values are defined as separate parameters in the configuration file. This approach ensures a clean separation of configuration rules from the core source data, avoiding inconsistent values within a single column of the static properties.

```yaml
surface_water-max_size-m3/day: 164383.56
desalination-max_size-m3/day: inf
groundwater-max_permit_surplus: 0.3
```

Save the data to file

In [ ]:
# Put everything in the StaticProperties object to save the data
dataf_dict = {str(source_type): df for source_type, df in sources_df.groupby('source_type')}
DROP_COLUMNS = ['source_type','counter', 'repurposed', 'replaces', 'effective_id']
for source_type, df in dataf_dict.items():
    df = df.reset_index(drop=True)
    
    columns2drop = DROP_COLUMNS
    if source_type != 'groundwater':
        columns2drop = columns2drop+['permit']
    if source_type != 'surface_water':
        columns2drop = columns2drop+['basin']
    df.drop(columns=columns2drop, inplace=True)
    dataf_dict[source_type] = df

    # Move source_id in the first column
    dataf_dict[source_type].set_index('source_id', inplace=True)
    dataf_dict[source_type].reset_index(inplace=True)

    # stringiy the time in the activation and closure dates
    dataf_dict[source_type]['activation_date'] = dataf_dict[source_type]['activation_date'].dt.strftime('%Y-%m-%d')
    dataf_dict[source_type]['closure_date'] = dataf_dict[source_type]['closure_date'].dt.strftime('%Y-%m-%d')

    # lose the decimal digits of capacitie and permits
    dataf_dict[source_type]['capacity-nominal'] = dataf_dict[source_type]['capacity-nominal'].apply(np.round).astype(float)
    if source_type == 'groundwater':
        dataf_dict[source_type]['permit'] = dataf_dict[source_type]['permit'].apply(np.round).astype(float)

sources_sps = StaticProperties('sources-static_properties', dataframes=dataf_dict)

sources_sps['global'] = global_df

sources_sps.dump(path='..')